#PRN=003

In [40]:
#to remove warnings
import warnings
warnings.filterwarnings('ignore')

In [41]:
#importing Library
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor



In [42]:
#loading dataset
big_data=pd.read_csv("D:\Machine Learning\ML_Class\Assignment\\train_dataset.csv")
big_data.isna().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [43]:
# dropping uncessary columns
data=big_data.drop(columns=['Item_Identifier','Outlet_Identifier'],axis=1)

In [ ]:
#Imputation of data
from sklearn.impute import SimpleImputer


impute_median=SimpleImputer(strategy='median')
impute_mode=SimpleImputer(strategy='most_frequent')
data['Item_Weight']=impute_median.fit_transform(data[['Item_Weight']]).ravel()#
data['Outlet_Size']=impute_mode.fit_transform(data[['Outlet_Size']])[:,0]


In [45]:
data.isna().sum()

Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

In [46]:
# Identify column types
X,y=data.drop('Item_Outlet_Sales',axis=1),data['Item_Outlet_Sales']
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [47]:
# Preprocessor: OneHot for categorical, scale numeric
preprocessor = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", StandardScaler(), numeric_cols)],
    remainder="drop")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
#making objects of models
from sklearn.linear_model import ElasticNet,LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

elastic=ElasticNet()

random=RandomForestRegressor(n_estimators=200,n_jobs=-1,random_state=42)

knr=KNeighborsRegressor(n_neighbors=5)

lr=LinearRegression()

gbr=GradientBoostingRegressor(random_state=42)

models=[['ElasticNet',elastic],
        ['RandomForestRegressor',random],
        ['KNeighborsRegressor',knr],
        ['Linear Regression',lr],
        ['GradientBoostingRegressor',gbr]]

In [50]:
#fitting and predicting for all models
scores=[]

for name,model in models:
    pipe=Pipeline(steps=[('prep',preprocessor),("model",model)])
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    scores.append([name,r2])

In [51]:
#displaying result
df_scores=pd.DataFrame(scores,columns=('Model_Name','R2_Score'))
df_scores.sort_values('R2_Score',ascending=False)

,Model_Name,R2_Score
4,GradientBoostingRegressor,0.602797
3,Linear Regression,0.579476
1,RandomForestRegressor,0.562163
2,KNeighborsRegressor,0.501087
0,ElasticNet,0.397712
